# DINOv2 ViT-B/14 — Full Training Script
- Architecture defined from scratch (no `torch.hub`)
- Weights from HuggingFace: `facebook/dinov2-base`

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from PIL import Image
from huggingface_hub import hf_hub_download

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

### SECTION 1 — DATA

In [ ]:
TRAIN_DIR = r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Projectt\CSE-281\dataset\train'

def remove_bad_images(root):
    removed = 0
    for folder, _, files in os.walk(root):
        for file in files:
            path = os.path.join(folder, file)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                os.remove(path)
                removed += 1
    print(f'Removed {removed} bad images')

remove_bad_images(TRAIN_DIR)

train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_train = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
full_val   = datasets.ImageFolder(root=TRAIN_DIR, transform=val_transforms)
print('Total images:', len(full_train))

targets = full_train.targets
indices = list(range(len(targets)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=42, stratify=targets
)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

train_dataset = Subset(full_train, train_idx)
val_dataset   = Subset(full_val,   val_idx)

train_targets  = [targets[i] for i in train_idx]
class_counts   = np.bincount(train_targets)
class_weights  = 1.0 / class_counts
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=16, sampler=sampler,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False,    num_workers=2, pin_memory=True)

### SECTION 2 — MODEL ARCHITECTURE (from scratch)

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3, embed_dim=768):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.Identity()

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return self.norm(x)

class LayerScale(nn.Module):
    def __init__(self, dim, init_values=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(init_values * torch.ones(dim))

    def forward(self, x):
        return x * self.gamma

class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = nn.GELU()
        self.fc2  = nn.Linear(hidden_features, in_features)
        self.drop = nn.Dropout(0.0)

    def forward(self, x):
        return self.drop(self.fc2(self.act(self.fc1(x))))

class Attention(nn.Module):
    def __init__(self, dim, num_heads=12):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = dim // num_heads
        self.scale     = self.head_dim ** -0.5
        self.qkv       = nn.Linear(dim, dim * 3)
        self.attn_drop = nn.Dropout(0.0)
        self.proj      = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(0.0)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = self.attn_drop(attn.softmax(dim=-1))
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))

class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, ls_init=1e-5):
        super().__init__()
        self.norm1      = nn.LayerNorm(dim, eps=1e-6)
        self.attn       = Attention(dim, num_heads)
        self.ls1        = LayerScale(dim, ls_init)
        self.drop_path1 = nn.Identity()
        self.norm2      = nn.LayerNorm(dim, eps=1e-6)
        self.mlp        = Mlp(dim, int(dim * mlp_ratio))
        self.ls2        = LayerScale(dim, ls_init)
        self.drop_path2 = nn.Identity()

    def forward(self, x):
        x = x + self.drop_path1(self.ls1(self.attn(self.norm1(x))))
        x = x + self.drop_path2(self.ls2(self.mlp(self.norm2(x))))
        return x

class DinoVisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=14, in_chans=3,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4.0):
        super().__init__()
        self.embed_dim   = embed_dim
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches      = self.patch_embed.num_patches
        self.cls_token   = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed   = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.blocks      = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio) for _ in range(depth)])
        self.norm        = nn.LayerNorm(embed_dim, eps=1e-6)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        B  = x.shape[0]
        x  = self.patch_embed(x)
        x  = torch.cat([self.cls_token.expand(B, -1, -1), x], dim=1)
        x  = x + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)[:, 0]   # CLS token → (B, 768)

### SECTION 3 — WEIGHT LOADING (HuggingFace, no `torch.hub`)

In [ ]:
def _remap_hf_state_dict(hf_sd):
    new_sd = {}
    for hf_key, v in hf_sd.items():
        key = hf_key[len("dinov2."):] if hf_key.startswith("dinov2.") else hf_key

        # patch embed
        if key == "embeddings.patch_embeddings.projection.weight": new_sd["patch_embed.proj.weight"] = v; continue
        if key == "embeddings.patch_embeddings.projection.bias":   new_sd["patch_embed.proj.bias"]   = v; continue
        # cls / pos
        if key == "embeddings.cls_token":          new_sd["cls_token"] = v; continue
        if key == "embeddings.position_embeddings": new_sd["pos_embed"] = v; continue
        # final norm
        if key == "layernorm.weight": new_sd["norm.weight"] = v; continue
        if key == "layernorm.bias":   new_sd["norm.bias"]   = v; continue

        if key.startswith("encoder.layer."):
            parts  = key.split(".")
            i, rest = parts[2], ".".join(parts[3:])
            pfx    = f"blocks.{i}."

            if rest == "norm1.weight": new_sd[pfx+"norm1.weight"] = v; continue
            if rest == "norm1.bias":   new_sd[pfx+"norm1.bias"]   = v; continue
            if rest == "norm2.weight": new_sd[pfx+"norm2.weight"] = v; continue
            if rest == "norm2.bias":   new_sd[pfx+"norm2.bias"]   = v; continue
            if rest == "layer_scale1.lambda1": new_sd[pfx+"ls1.gamma"] = v; continue
            if rest == "layer_scale2.lambda1": new_sd[pfx+"ls2.gamma"] = v; continue
            if rest == "attention.output.dense.weight": new_sd[pfx+"attn.proj.weight"] = v; continue
            if rest == "attention.output.dense.bias":   new_sd[pfx+"attn.proj.bias"]   = v; continue
            if rest == "intermediate.dense.weight": new_sd[pfx+"mlp.fc1.weight"] = v; continue
            if rest == "intermediate.dense.bias":   new_sd[pfx+"mlp.fc1.bias"]   = v; continue
            if rest == "output.dense.weight": new_sd[pfx+"mlp.fc2.weight"] = v; continue
            if rest == "output.dense.bias":   new_sd[pfx+"mlp.fc2.bias"]   = v; continue
            if rest in ("attention.attention.query.weight", "attention.attention.key.weight",
                        "attention.attention.value.weight", "attention.attention.query.bias",
                        "attention.attention.key.bias",     "attention.attention.value.bias"):
                new_sd[f"__qkv__{i}__{rest}"] = v; continue

    # fuse Q / K / V → single qkv
    for i in {k.split("__")[2] for k in new_sd if k.startswith("__qkv__")}:
        pfx = f"__qkv__{i}__attention.attention."
        new_sd[f"blocks.{i}.attn.qkv.weight"] = torch.cat([new_sd.pop(pfx+"query.weight"), new_sd.pop(pfx+"key.weight"),  new_sd.pop(pfx+"value.weight")], dim=0)
        new_sd[f"blocks.{i}.attn.qkv.bias"]   = torch.cat([new_sd.pop(pfx+"query.bias"),   new_sd.pop(pfx+"key.bias"),    new_sd.pop(pfx+"value.bias")],   dim=0)

    return new_sd

def load_dinov2_vitb14_weights(model):
    print("Downloading facebook/dinov2-base weights from HuggingFace …")
    weight_path = hf_hub_download(repo_id="facebook/dinov2-base", filename="pytorch_model.bin")
    hf_sd  = torch.load(weight_path, map_location="cpu")
    our_sd = _remap_hf_state_dict(hf_sd)
    missing, unexpected = model.load_state_dict(our_sd, strict=True)
    if missing:    print(f"[WARN] Missing:    {missing[:3]}")
    if unexpected: print(f"[WARN] Unexpected: {unexpected[:3]}")
    print("✅  Backbone weights loaded.")
    return model

### SECTION 4 — CLASSIFIER

In [ ]:
class DINOv2Classifier(nn.Module):
    def __init__(self, num_classes=17, pretrained=True):
        super().__init__()
        self.backbone   = DinoVisionTransformer()
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(768, num_classes),
        )
        if pretrained:
            load_dinov2_vitb14_weights(self.backbone)

    def forward(self, x):
        return self.classifier(self.backbone(x))

model = DINOv2Classifier(num_classes=17, pretrained=True).to(device)

# ── freeze everything; only train the head first ──
for param in model.backbone.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total      = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}')

### SECTION 5 — TRAINING LOOP (dynamic unfreeze on plateau)

In [ ]:
NUM_EPOCHS = 50
patience   = 2   # epochs without improvement before unfreezing next group

# Grouped from outer → inner so we peel back the backbone layer by layer
layers_to_unfreeze = [
    [model.backbone.norm],                                          # group 0: final norm
    [model.backbone.blocks[11], model.backbone.blocks[10]],        # group 1: last 2 blocks
    [model.backbone.blocks[9],  model.backbone.blocks[8]],         # group 2
    [model.backbone.blocks[7],  model.backbone.blocks[6]],         # group 3
    [model.backbone.blocks[5],  model.backbone.blocks[4],
     model.backbone.blocks[3],  model.backbone.blocks[2],
     model.backbone.blocks[1],  model.backbone.blocks[0],
     model.backbone.patch_embed],                                   # group 4: the rest
]

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = AdamW(model.classifier.parameters(), lr=5e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

best_val_acc  = 0.0
best_val_loss = float('inf')
plateau_counter  = 0
unfreeze_index   = 0

print(f"\n{'Epoch':>5} | {'Train Loss':>10} | {'Train BAcc':>10} | {'Val Loss':>8} | {'Val BAcc':>8} | {'Best':>6}")
print("-" * 65)

for epoch in range(NUM_EPOCHS):

    # ── TRAIN ──────────────────────────────────────────────
    model.train()
    running_loss, all_preds, all_labels = 0.0, [], []

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss = running_loss / len(train_loader.dataset)
    train_acc  = balanced_accuracy_score(all_labels, all_preds)

    # ── VALIDATE ───────────────────────────────────────────
    model.eval()
    val_loss, val_preds, val_labels = 0.0, [], []

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            val_loss += criterion(outputs, labels).item() * images.size(0)
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_loader.dataset)
    val_acc  = balanced_accuracy_score(val_labels, val_preds)

    scheduler.step()

    # ── BEST MODEL TRACKING ────────────────────────────────
    is_best = False
    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        is_best = True
    elif val_acc == best_val_acc and val_loss < best_val_loss:
        best_val_loss = val_loss
        is_best = True

    if is_best:
        plateau_counter = 0
        torch.save(model.state_dict(), r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Projectt\CSE-281\eyad\are_you_dino.pth')
    else:
        plateau_counter += 1

    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>10.4f} | {val_loss:>8.4f} | {val_acc:>8.4f} | {'✓' if is_best else f'Plateau: {plateau_counter}/{patience}'}")

    # ── DYNAMIC UNFREEZE TRIGGER ───────────────────────────
    if plateau_counter >= patience and unfreeze_index < len(layers_to_unfreeze):
        print(f"\n🔥 Plateau! Unfreezing backbone group {unfreeze_index + 1}/{len(layers_to_unfreeze)} 🔥")

        for module in layers_to_unfreeze[unfreeze_index]:
            for param in module.parameters():
                param.requires_grad = True

        unfreeze_index  += 1
        plateau_counter  = 0

        print("   -> Resetting SGD optimizer with differential learning rates.")
        optimizer = torch.optim.SGD([
            {'params': model.classifier.parameters(), 'lr': 1e-2},
            {'params': [p for n, p in model.named_parameters() if 'classifier' not in n and p.requires_grad], 'lr': 1e-3},
        ], momentum=0.9, weight_decay=1e-4, nesterov=True)

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-5)

print(f'\nBest Val Balanced Accuracy: {best_val_acc:.4f}  |  Best Val Loss: {best_val_loss:.4f}')

### SECTION 6 — INFERENCE & SUBMISSION

In [ ]:
import pandas as pd
from torch.utils.data import Dataset

class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        self.transform   = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.folder_path, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            print(f"  [!] Warning: {img_name} is corrupted. Using blank placeholder.")
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, img_name

test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

TEST_DIR     = r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Projectt\CSE-281\dataset\test'
test_dataset = TestDataset(folder_path=TEST_DIR, transform=test_transforms)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

# Load best checkpoint
model.load_state_dict(torch.load(r'C:\LOCAL DISK D\Eyad\Faculty assignemts\Sem 4\Intro to AI\Projectt\CSE-281\eyad\are_you_dino.pth', map_location=device))
model.to(device)
model.eval()

predictions = []
print(f"\nRunning inference on {len(test_dataset)} test images …")

with torch.no_grad():
    for i, (images, img_names) in enumerate(test_loader):
        images   = images.to(device)
        outputs  = model(images)
        _, preds = torch.max(outputs, 1)
        for img_name, pred in zip(img_names, preds):
            predictions.append({'ImageName': img_name, 'ClassLabel': pred.item()})
        if (i + 1) % 25 == 0:
            print(f"  Processed batch {i+1}/{len(test_loader)} …")

submission_df = pd.DataFrame(predictions)
submission_df['sort_number'] = submission_df['ImageName'].str.extract(r'(\d+)').astype(int)
submission_df = submission_df.sort_values('sort_number').drop(columns=['sort_number'])
submission_df.to_csv('Eyad_model_submission_V7.csv', index=False)
print("✅  Saved: Eyad_model_submission_V7.csv")